In [61]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = SparkSession.builder.appName("Review").master("yarn").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/21 15:15:27 WARN Utils: Your hostname, Parashus-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.2.130 instead (on interface en0)
26/02/21 15:15:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/21 15:15:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/02/21 15:15:29 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [3]:
spark

In [ ]:
# Objective:
# Perform ETL and analytics using PySpark and Hadoop (HDFS).
# Step-by-Step Instructions:
# Step 1: Upload Data to HDFS
# Create folder: /ecommerce/raw/
# Upload orders.csv and order_items.csv
# Step 2: Read Data in PySpark
# Load both CSV files as DataFrames
# Print schema
# Count total records
# Step 3: Join Data
# Join orders and order_items using order_id
# Handle null values
# Convert date columns to proper format
# Step 4: Perform Analytics
# Calculate total revenue
# Revenue per month
# Top 5 selling products
# Average order value
# Step 5: Save Output
# Store results in Parquet format
# Path: /ecommerce/analytics/

### Perform ETL and analytics using PySpark and Hadoop (HDFS).

In [55]:
# Step 1: Upload Data to HDFS
!hdfs dfs -mkdir /ecommerce
!hdfs dfs -mkdir /ecommerce/raw
!hdfs dfs -mkdir /ecommerce/analytics

# Upload orders.csv and order_items.csv
!hdfs dfs -put data/order_items.csv /ecommerce/raw/
!hdfs dfs -put data/orders.csv /ecommerce/raw/

2026-02-21 16:30:58,433 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
mkdir: `/ecommerce': File exists
2026-02-21 16:30:59,941 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
mkdir: `/ecommerce/raw': File exists
2026-02-21 16:31:01,199 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-02-21 16:31:02,431 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
put: `/ecommerce/raw/order_items.csv': File exists
2026-02-21 16:31:03,552 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
put: `/ecommerce/raw/orders.csv': File exists


### Step 2: Read Data in PySpark

In [16]:
# Load both CSV files as DataFrames
# Print schema
# Count total records
orders = spark.read.csv("hdfs:///ecommerce/raw/orders.csv",\
                        inferSchema="True",\
                       header=True\
                       )
order_items = spark.read.csv("hdfs:///ecommerce/raw/order_items.csv",\
                             inferSchema=True,\
                            header=True\
                            )
orders.printSchema()
order_items.printSchema()

print("Number of records in orders: ", orders.count())
print("Number of records in orders: ", order_items.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)

Number of records in orders:  99441
Number of records in orders:  112650


In [40]:
# Step 3: Join Data
# Join orders and order_items using order_id
# Handle null values
# Convert date columns to proper format

full_data = orders.join(
    order_items,
    "order_id",
    "inner"
)
full_data.printSchema()

full_data_cleaned = full_data.dropna(subset=[
    "order_id",
    "Price"
]
)

full_data_cleaned = full_data_cleaned.withColumn("order_purchase_timestamp",
                                                to_date(col("order_purchase_timestamp"))
                                                )\
.withColumn("order_delivered_carrier_date",
            to_date(col("order_delivered_carrier_date"))
           )\
.withColumn("order_delivered_customer_date",
            to_date(col("order_delivered_customer_date"))
           )\
.withColumn("order_estimated_delivery_date",
            to_date(col("order_estimated_delivery_date"))
           )
full_data_cleaned.show(1)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



[Stage 88:>                                                         (0 + 2) / 2]

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|order_item_id|          product_id|           seller_id|shipping_limit_date|price|freight_value|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+-------------+--------------------+--------------------+-------------------+-----+-------------+
|00018f77f2f0320c5...|f6dd3ec061db4e398...|   delivered|              2017-04-26|2017-04-26 11:05:13|                  2017-05-04|         

In [63]:
# Step 4: Perform Analytics
# Calculate total revenue
# Revenue per month
# Top 5 selling products
# Average order value
full_data_cleaned = full_data_cleaned.withColumn("Total price",
                                                col("price")+col("freight_value"))
                                                

# Calculate total revenue
print("Total Revenue")
total_revenue = full_data_cleaned.agg(sum("Total price").alias("Total Revenue"))
total_revenue = total_revenue.withColumn("Total Revenue", col("Total Revenue").cast(DecimalType(10,2)))
total_revenue.show()

# Revenue per month
print("Monthly Revenue")
full_data_cleaned = full_data_cleaned.withColumn("order month",
                                                date_format("order_purchase_timestamp","yyyy-MM")
                                                )
monthly_revenue = full_data_cleaned.groupBy("order month").agg(sum("Total price").alias("Monthly Revenue")) 
monthly_revenue.show()

# Top 5 selling products
print("Top 5 selling products")
top_selling_products = full_data_cleaned.groupBy("product_id")\
    .agg(count("product_id")\
        .alias("total_sold"))\
.orderBy(col("total_sold").desc())
top_selling_products.limit(5).show()

# Average order value
print("Average Order value")
order_value = full_data_cleaned.groupBy("order_id").agg(sum("Total price").alias("Order_value"))
average_order_value = order_value.agg(
    avg("Order_value").alias("average_order_value")
)
average_order_value.show()

Total Revenue
+-------------+
|Total Revenue|
+-------------+
|  15843553.24|
+-------------+

Monthly Revenue
+-----------+------------------+
|order month|   Monthly Revenue|
+-----------+------------------+
|    2017-09| 720398.9100000022|
|    2017-10| 769312.3700000022|
|    2017-05| 586190.9500000016|
|    2018-06|1022677.1099999979|
|    2017-11|1179143.7700000012|
|    2018-03|        1155126.82|
|    2018-02| 986908.9600000011|
|    2017-03| 432048.5900000002|
|    2016-12|             19.62|
|    2017-08| 668204.6000000034|
|    2016-09|            354.75|
|    2017-06| 502963.0400000009|
|    2016-10|56808.839999999975|
|    2017-02|286280.61999999965|
|    2017-04|412422.24000000034|
|    2018-05|1149781.8200000026|
|    2018-08|1003308.4699999981|
|    2017-07| 584971.6200000023|
|    2017-12| 863547.2300000025|
|    2018-04|1159698.0400000012|
+-----------+------------------+
only showing top 20 rows
Top 5 selling products
+--------------------+----------+
|          prod

In [66]:
# Step 5: Save Output
# Store results in Parquet format
# Path: /ecommerce/analytics/
full_data_cleaned.write.mode("overwrite").parquet("hdfs:///ecommerce/analytics/full_data_cleaned")
total_revenue.write.mode("overwrite").parquet("hdfs:///ecommerce/analytics/total_revenue")
monthly_revenue.write.mode("overwrite").parquet("hdfs:///ecommerce/analytics/monthly_revenue")
top_selling_products.write.mode("overwrite").parquet("hdfs:///ecommerce/analytics/top_selling_products")
average_order_value.write.mode("overwrite").parquet("hdfs:///ecommerce/analytics/average_order_value")